# Frame Your Lane as an ML Task

**User Intent Lane**

This notebook maps my chosen lane onto the ML loop.

## 1. My Lane as an ML Task

**Task Type:** Classification (Binary)

**Explanation:**  
I want to predict whether a page matches user intent based on engagement signals. 

Specifically, I'll start with a binary classification:
- **1 = High Engagement Intent** (user found what they wanted)
- **0 = Low Engagement Intent** (user didn't find what they wanted)

In the future, this could expand to multi-class: Informational / Navigational / Transactional.

## 2. Target or Proxy

**Target:** User intent type

**Proxy (since explicit intent labels don't exist):**  
I'll use engagement signals as a proxy for intent:
- **High engagement (intent matched):** CTR > 0.05 AND scroll_rate > 0.3
- **Low engagement (intent didn't match):** CTR <= 0.05 OR scroll_rate <= 0.3

**Justification:**  
If a user clicks and scrolls deeply, they likely found what they wanted.  
If they don't click or scroll, the content probably didn't match their intent.

## 3. Success Metric

**Primary Metric:** Precision@50

**Why:**  
Of the top 50 pages our model flags as "high intent," how many actually have high engagement?

**Baseline to Beat:**  
Hand-rule from notebook 02: Precision@50 = 0.680

**What "Good" Looks Like:**
- Precision@50 > 0.700
- Interpretable model (decision tree with depth ≤ 5)
- No feature leakage

**Secondary Metrics:**
- Recall@50
- F1-Score
- AUC-ROC

## 4. Unit of Analysis (as a Real Dataframe)

In [ ]:
import pandas as pd
import numpy as np

# Load the data
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

print("First 5 rows of data:")
display(df.head())

print(f"\nShape: {df.shape} (rows x columns)")
print(f"Columns: {df.columns.tolist()}")

**What this shows:**
- One row = one page (content_id)
- Engagement metrics: impressions, clicks, CTR, scroll_rate
- Content features: age, type, word_count
- Position metrics: avg_position

**The Target Column:**
I'll create a target column called `high_intent_label` based on engagement proxies:

In [ ]:
# Create proxy target
df['high_intent_label'] = ((df['ctr'] > 0.05) & (df['scroll_rate'] > 0.3)).astype(int)

print("Target distribution:")
print(df['high_intent_label'].value_counts())
print(f"\nPercentage high intent: {df['high_intent_label'].mean()*100:.1f}%")

## 5. Why ML Beats a Fixed Rule Here

**A Fixed Rule Would Be:**
"If CTR > 0.05 and scroll_rate > 0.3, then high intent"

**Problems with This Rule:**
1. **Too simple:** Misses complex patterns
2. **Static:** Can't adapt to changing user behavior
3. **Misses edge cases:** Some high-intent pages might have low CTR (brand queries)
4. **Ignores feature interactions:** Position + content_age + word_count together might tell a richer story

**Why ML is Better:**
1. **Learns patterns:** Finds combinations of features that indicate intent
2. **Adapts:** Can be retrained with new data
3. **Generalizes:** Works on unseen data
4. **Quantifies uncertainty:** Gives probability scores, not just binary decisions

**Real-world Example:**
Two pages both have CTR = 0.04:
- Page A: avg_position = 2.5, content_age = 10 days, scroll_rate = 0.4 → ML might say HIGH intent
- Page B: avg_position = 15.0, content_age = 90 days, scroll_rate = 0.1 → ML might say LOW intent

A fixed rule would treat them the same. ML captures the nuance.

## 6. Self-Check

✅ I've named the task type: Classification (Binary)

✅ I've defined the target/proxy: high_intent_label based on CTR + scroll_rate

✅ I've chosen a success metric: Precision@50

✅ I've shown the unit of analysis: one row = one page (content_id)

✅ I've explained why ML beats a fixed rule: captures complex patterns

✅ My approach is cautious and directional, not claiming causation

✅ I've identified the baseline to beat: hand-rule (0.680 Precision@50)

---

**Provisional Caveat:**
> *"This framing is provisional. As I explore the data, I may refine my target definition, feature set, or task type. The goal is to demonstrate a thoughtful approach, not a perfect solution."*